# 02 – Tiền xử lý và phát hành dữ liệu cho mô hình

Notebook của M1, làm các task **M1-05** (làm sạch), **M1-06** (chiếu toạ độ), **M1-07** (đặc trưng thời gian), **M1-08** (kiểm tra và bàn giao).

Đầu vào: file thô do notebook 01 tải về (`data/raw/`). Notebook này **không gọi cổng dữ liệu**, nên chạy được cả trên máy cá nhân lẫn Colab, miễn là có file thô.

Đầu ra:

| File | Nội dung | Ai dùng |
|---|---|---|
| `data/processed/master_pickups_<phiên bản>.parquet` | Mọi dòng sau làm sạch | M1, ai cần toàn bộ dữ liệu |
| `data/processed/model_input_sample_<phiên bản>.parquet` | Mẫu phân tầng thứ × giờ | **M2, M3** (qua `paths.model_input` trong config) |
| `data/manifests/*_manifest.json` | Cách tạo, số dòng, sha256 | M2, M3, M4 |
| `outputs/metrics/preprocess/` | Cleaning log, kiểm tra CRS, từ điển đặc trưng, kết quả QA | Báo cáo |

Ba nguyên tắc:
- Ma trận đặc trưng của cả ba mô hình **chỉ gồm `[x_m, y_m]`**. Cột thời gian chỉ để lọc lát cắt.
- Không loại dòng âm thầm: mỗi quy tắc có số dòng trước và sau trong cleaning log.
- Không ghi đè bản đã phát hành: muốn đổi dữ liệu thì tăng `release.version` trong config.

In [11]:
# ===== O THIET LAP (dua tren o dung chung cua notebook 00; them BRANCH va noi Drive) =====
# Colab: clone repo, cai thu vien, noi data/ voi Google Drive.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
BRANCH = "m1/extract-eda"        # doi thanh "main" sau khi nhanh M1 duoc merge
DRIVE_DIR = "KhaiPhaDuLieu"      # = MyDrive/KhaiPhaDuLieu/data
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if IN_COLAB:
    from src.colab import link_drive_data
    print("data/ ->", link_drive_data(DRIVE_DIR))   # file ghi vao data/ se nam luon tren Drive

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

Colab: False | Goc repo: C:\Users\admin\Downloads\KhaiPhaDuLieu_final\KhaiPhaDuLieu


## 1. Tham số

Tất cả lấy từ `config/experiment.yaml` và `config/cleaning_rules.yaml`; notebook không có số nào gõ cứng.

In [12]:
import json
import pandas as pd
from src.config import load_config, resolve_path
from src.data import extract as E
from src.data import preprocess as P

cfg = load_config()
rules = P.load_rules()
START, END = cfg["extract"]["start"], cfg["extract"]["end"]
CRS, SEED = cfg["project"]["crs"], cfg["project"]["seed"]
VERSION, SAMPLE_N = cfg["release"]["version"], cfg["sampling"]["target_size"]
SLICES = cfg["time_slices"]

RAW_FILE = resolve_path(cfg, "raw_dir") / f"{E.raw_name(START, END)}.parquet"
RAW_MAN = resolve_path(cfg, "manifests_dir") / f"{E.raw_name(START, END)}_manifest.json"
PROC_DIR, MAN_DIR = resolve_path(cfg, "processed_dir"), resolve_path(cfg, "manifests_dir")
REPORT_DIR = resolve_path(cfg, "preprocess_dir")
NAMES = P.release_names(VERSION)
SAMPLE_FILE, MASTER_FILE = PROC_DIR / NAMES["sample"], PROC_DIR / NAMES["master"]
SAMPLE_MAN, MASTER_MAN = MAN_DIR / NAMES["manifest"], MAN_DIR / NAMES["master_manifest"]
assert SAMPLE_FILE == resolve_path(cfg, "model_input") and SAMPLE_MAN == resolve_path(cfg, "manifest"), \
    "paths.model_input / paths.manifest trong config khong khop release.version"

HAVE_RAW = RAW_FILE.exists()
print("Cua so      :", START, "..", END, "| CRS:", CRS, "| seed:", SEED)
print("Phien ban   :", VERSION, "| co mau muc tieu:", f"{SAMPLE_N:,}")
print("File tho    :", RAW_FILE.name, "| da co:", HAVE_RAW)
print("Da phat hanh:", SAMPLE_FILE.exists())
if not HAVE_RAW:
    print("CHUA CO FILE THO. Chay notebook 01 tren Colab, hoac tai thu muc raw/ va manifests/ tu Drive ve data/.")

Cua so      : 2023-04-17 .. 2023-05-14 | CRS: EPSG:26916 | seed: 42
Phien ban   : v1 | co mau muc tieu: 100,000
File tho    : raw_extract_20230417_20230514.parquet | da co: False
Da phat hanh: False
CHUA CO FILE THO. Chay notebook 01 tren Colab, hoac tai thu muc raw/ va manifests/ tu Drive ve data/.


## 2. Làm sạch và gắn cờ (M1-05)

Quy tắc nằm trong `config/cleaning_rules.yaml`, chia hai loại:
- **drop**: loại dòng khi không xác định được "một lượt đón, ở đâu, lúc nào" (thiếu mã, thiếu thời điểm, thiếu hoặc sai toạ độ). Áp dụng lần lượt theo thứ tự.
- **flag**: giữ dòng và thêm cột `flag_*` khi trường đo chuyến đi bất thường (thời lượng 0, quãng đường 0, tốc độ phi lý…). Các trường này không nằm trong đặc trưng, nên không có lý do loại một lượt đón vì chúng.

Bảng dưới là **cleaning log**: mỗi quy tắc một dòng, có số dòng trước, số dòng dính quy tắc và số dòng còn lại.

In [13]:
if HAVE_RAW:
    raw_man = json.loads(RAW_MAN.read_text(encoding="utf-8"))
    assert E.sha256(RAW_FILE) == raw_man["sha256"], "file tho khac manifest: khong dung file nay"
    typed = E.parse_types(pd.read_parquet(RAW_FILE))
    master, log = P.build_master(typed, rules, SLICES, START, END, CRS)   # lam sach + dac trung + chieu toa do
    print(f"Tho: {len(typed):,} dong -> sau lam sach: {len(master):,} dong "
          f"(loai {len(typed) - len(master):,}, {1 - len(master) / len(typed):.2%})")
    display(log[["step", "rule_id", "action", "rows_before", "rows_matched", "rows_after", "matched_pct_of_raw"]])
    print("Do nhay cua nguong toc do (so dong vuot nguong; khong loai dong nao):")
    display(P.speed_sensitivity(master, rules))

## 3. Chiếu toạ độ sang mét (M1-06)

Kinh độ và vĩ độ tính bằng độ, và ở vĩ độ Chicago một độ kinh chỉ dài bằng khoảng 74% một độ vĩ, nên không thể tính khoảng cách Euclid trực tiếp trên lat/lon. `project_coordinates()` đổi sang hệ **EPSG:26916** (UTM múi 16 Bắc), đơn vị mét, tạo ra `x_m`, `y_m`.

Ba phép kiểm tra: vùng áp dụng của hệ chiếu có phủ hết điểm không; chiếu ngược về lat/lon lệch bao nhiêu; khoảng cách trên `x/y` lệch bao nhiêu so với khoảng cách trắc địa thật.

In [14]:
if HAVE_RAW:
    crs_info = P.validate_crs(master, CRS)
    assert crs_info["all_points_in_area_of_use"] and crs_info["unit"] == "metre"
    assert crs_info["roundtrip_max_error_m"] < 0.01, "chieu nguoc lech qua 1 cm"
    assert crs_info["distance_max_rel_error_pct"] < 0.1, "khoang cach lech qua 0,1%"
    display(pd.Series({k: v for k, v in crs_info.items() if k != "area_of_use"}, name="gia_tri").to_frame())

## 4. Đặc trưng thời gian (M1-07)

- `pickup_hour`, `weekday` (0 = thứ Hai), `is_weekend`, `is_weekday`.
- `daypart`: AM_PEAK 07–09 giờ, MIDDAY 10–15 giờ, PM_PEAK 16–18 giờ, NIGHT 19–06 giờ; không phụ thuộc thứ.
- `time_slice`: nhóm để chạy mô hình độc lập, theo `time_slices` trong config. Dòng không thuộc lát cắt nào mang nhãn `OTHER` và vẫn được giữ.

Bảng chéo dưới cho thấy hai cột này liên hệ thế nào. Các mốc biên (06:59/07:00, 09:59/10:00, thứ Sáu/thứ Bảy) được kiểm tra trong `tests/test_preprocess.py`.

In [15]:
if HAVE_RAW:
    display(pd.crosstab(master["time_slice"], master["daypart"], margins=True, margins_name="Tong"))
    display(P.coord_summary(master))

## 5. Lấy mẫu phân tầng

Mẫu lấy từ master, phân tầng theo **thứ × giờ** (168 tầng): mỗi tầng được chia số dòng theo đúng tỷ trọng của nó trong master, nên mẫu giữ nguyên nhịp ngày và nhịp tuần. Cùng master, cùng cỡ mẫu, cùng seed thì ra đúng cùng một tập `trip_id`.

Cỡ mẫu lấy từ `sampling.target_size` trong config. Bảng dưới so tỷ trọng từng lát cắt giữa mẫu và master (`diff_pp` là chênh lệch tính bằng điểm phần trăm).

In [16]:
if HAVE_RAW:
    sample = P.stratified_sample(master, SAMPLE_N, seed=SEED)
    print(f"Mau: {len(sample):,} / {len(master):,} dong ({len(sample) / len(master):.1%})")
    display(P.sample_check(master, sample, ["time_slice"]))
    display(P.coord_summary(sample))

## 6. Phát hành (M1-08)

Ghi master, mẫu và hai manifest. Nếu phiên bản này **đã được ghi** thì notebook không ghi lại; thay vào đó nó so kết quả vừa tính với file trên đĩa để xác nhận quy trình tái lập được.

In [17]:
if HAVE_RAW:
    if SAMPLE_FILE.exists():
        old = pd.read_parquet(SAMPLE_FILE)
        same = old["trip_id"].equals(sample["trip_id"]) and (old[P.FEATURES].to_numpy() == sample[P.FEATURES].to_numpy()).all()
        assert same, (f"Ket qua vua tinh KHAC ban {VERSION} da phat hanh. Neu co y doi du lieu / quy tac / co mau "
                      "thi tang release.version trong config; khong ghi de ban cu.")
        print(f"Ban {VERSION} da phat hanh tu truoc; tinh lai ra dung cung du lieu. Khong ghi lai.")
    else:
        P.release(master, sample, log, raw_man, PROC_DIR, MAN_DIR, VERSION, SEED, CRS, START, END, SAMPLE_N)
        print(f"Da phat hanh ban {VERSION}.")
    man = json.loads(SAMPLE_MAN.read_text(encoding="utf-8"))
    for k in ["sample_id", "file", "rows_raw", "rows_after_cleaning", "rows_in_sample", "rows_by_time_slice",
              "unique_pickup_coords", "crs", "seed", "sha256"]:
        print(f"{k:22s}: {man[k]}")

## 7. Kiểm tra file đã phát hành

`validate_release()` **đọc lại file từ đĩa** (không dùng bảng trong bộ nhớ) và kiểm tra từng điều kiện nghiệm thu: schema, kiểu dữ liệu, giá trị rỗng, trùng `trip_id`, phép chiếu, phạm vi, lát cắt, số dòng và sha256 so với manifest. Có một kiểm tra không đạt thì notebook dừng.

In [18]:
if HAVE_RAW:
    qa = P.validate_release(SAMPLE_FILE, SAMPLE_MAN, SLICES, master_path=MASTER_FILE, crs=CRS, bbox=rules["bbox"])
    qa_master = P.validate_release(MASTER_FILE, MASTER_MAN, SLICES, crs=CRS, bbox=rules["bbox"])
    display(qa)
    print(f"Mau   : {int(qa.ok.sum())}/{len(qa)} kiem tra dat")
    print(f"Master: {int(qa_master.ok.sum())}/{len(qa_master)} kiem tra dat")
    assert qa.ok.all() and qa_master.ok.all(), "CO KIEM TRA KHONG DAT - khong giao file nay"
    reports = P.write_reports(REPORT_DIR, log, rules, master, sample, crs_info, qa)
    for name, path in reports.items():
        print(f"{name:28s} {path.relative_to(ROOT)}")

## 8. Cách M2 và M3 đọc dữ liệu

Trong notebook mô hình, đổi `DATA_KEY = "model_input"`. Đoạn dưới là đúng cách đọc đã thống nhất trong `docs/dinh-dang-du-lieu.md`, chạy thử trên file vừa phát hành.

In [19]:
if HAVE_RAW:
    df = pd.read_parquet(resolve_path(cfg, "model_input"))
    for s in SLICES:
        part = df[df["time_slice"] == s]                      # giu nguyen thu tu trip_id
        X = part[cfg["project"]["features"]].to_numpy(dtype="float64")
        print(f"{s:11s} X.shape = {X.shape} | toa do khac nhau: {len(pd.DataFrame(X).drop_duplicates()):,}")

## 9. Chép báo cáo về Drive (chỉ khi chạy trên Colab)

Trên Colab, `data/` đã là thư mục Drive nên file parquet và manifest nằm sẵn ở đó. Riêng `outputs/` sẽ mất khi đóng phiên, nên chép sang `KhaiPhaDuLieu/data/preprocess_outputs`. Chạy trên máy cá nhân thì ô này không làm gì.

In [20]:
import shutil

if HAVE_RAW and IN_COLAB:
    dst = PROC_DIR.parent / "preprocess_outputs"
    shutil.copytree(REPORT_DIR, dst, dirs_exist_ok=True)
    print("Da chep ve Drive:", dst)